# Nyfasi Deluxe Detangler: TRELLIS.2 image-to-3D GPU run

Run this notebook in Google Colab with a Linux NVIDIA GPU exposing **at least 24 GB VRAM** (for example, an eligible L4/A100 runtime). GPU allocation is subject to your Colab plan and availability. This notebook does not bypass quotas or session limits.

It downloads the five attributed research references from the project source script, lets you select one image, runs Microsoft's TRELLIS.2-4B, and downloads a textured GLB. The generated mesh is a visual hypothesis, not a measured or dimensionally accurate CAD model. Do not redistribute the source photographs without permission.

In [ ]:
import torch, subprocess, os, platform
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / (1024**3)
print('GPU:', props.name)
print(f'VRAM: {vram_gib:.1f} GiB')
if vram_gib < 23.5:
    raise RuntimeError(f'TRELLIS.2 needs at least 24 GB VRAM; this runtime has {vram_gib:.1f} GiB. Try an eligible L4/A100 runtime.')
!nvidia-smi

## 1. Get the project and reference images
The branch below is the open implementation PR. Change it only if you intentionally want a different revision.

In [ ]:
PROJECT_BRANCH = 'feat/nyfasi-reference-images-trellis2'
!git clone --depth 1 --branch $PROJECT_BRANCH https://github.com/andiekobbietks/product-engineering-lab.git /content/product-engineering-lab
%cd /content/product-engineering-lab
!python -m pip install -q Pillow
!python experiments/001-nyfasi-reconstruction/scripts/download_reference_images.py


In [ ]:
import json
from pathlib import Path
ref_dir = Path('experiments/001-nyfasi-reconstruction/output/reference-images')
manifest = json.loads((ref_dir/'manifest.json').read_text())
for i, item in enumerate(manifest['sources']):
    print(f"{i}: {item['id']} | {item['filename']} | {item['width_px']}x{item['height_px']} | {item['attribution']}")
print('\nSelect an image index. Prefer a single clear product view; avoid lifestyle images with hands or props.')
image_index = 0
assert 0 <= image_index < len(manifest['sources'])
selected = manifest['sources'][image_index]
input_image = ref_dir / selected['filename']
print('Selected:', selected['id'], input_image)

## 2. Install Microsoft's TRELLIS.2 environment
This compiles CUDA extensions and downloads large dependencies. It can take considerable time. The official project targets Linux, CUDA 12.4 and NVIDIA GPUs with at least 24 GB VRAM.

In [ ]:
%cd /content
!git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive
%cd /content/TRELLIS.2
!bash -lc 'source ./setup.sh --new-env --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm'

## 3. Generate the 3D hypothesis
Run inference using the repository's export script. TRELLIS.2 generates plausible geometry and materials from one image; it cannot infer true hidden geometry or reliable physical scale from a single photograph.

In [ ]:
%cd /content/product-engineering-lab
!cp "$input_image" /content/nyfasi_input.png
!cp experiments/001-nyfasi-reconstruction/scripts/run_trellis2_image_to_3d.py /content/TRELLIS.2/run_nyfasi_trellis2.py
%cd /content/TRELLIS.2
!bash -lc 'source /opt/conda/etc/profile.d/conda.sh 2>/dev/null || true; conda activate trellis2 && python run_nyfasi_trellis2.py /content/nyfasi_input.png --output /content/nyfasi_trellis2_hypothesis.glb'
from pathlib import Path
glb = Path('/content/nyfasi_trellis2_hypothesis.glb')
assert glb.exists() and glb.stat().st_size > 100_000, 'GLB output missing or unexpectedly small'
print(f'GLB generated: {glb.stat().st_size/1024/1024:.2f} MiB')

## 4. Download the result and its provenance
Keep the input image ID, original URL, SHA-256 and model name with the generated file. The mesh must not replace the parametric Sand3D model or its evidence-based dimension ledger.

In [ ]:
import shutil
shutil.copy2(glb, '/content/nyfasi_trellis2_hypothesis.glb')
provenance = {
  'model': 'microsoft/TRELLIS.2-4B',
  'input_reference_id': selected['id'],
  'input_source_url': selected['url'],
  'input_sha256': selected['sha256'],
  'input_attribution': selected['attribution'],
  'input_rights_note': selected['rights'],
  'output_file': glb.name,
  'interpretation': 'AI-generated visual hypothesis; not dimensionally calibrated or verified CAD geometry.'
}
Path('/content/nyfasi_trellis2_provenance.json').write_text(json.dumps(provenance, indent=2)+'\\n')
from google.colab import files
files.download(str(glb))
files.download('/content/nyfasi_trellis2_provenance.json')